# Notebook 50 — Side-Balanced Policy Fine-Tuning

## Purpose

#### This notebook fine-tunes a policy model using the corrected expert-policy dataset generated by Notebook 49.

#### The training pipeline will:

1. Load and validate the Notebook 49 handoff artifacts.
2. inspect policy-class and side distributions.
3. construct side-aware training, validation, and test splits.
4. encode battle-state features and expert actions.
5. train a weighted policy classifier.
6. evaluate overall and side-specific performance.
7. compare the fine-tuned policy against the previous policy.
8. export the trained model and Notebook 50 handoff artifacts.

## Notebook 49 handoff

#### Expected upstream status:

- Final status: `READY_FOR_POLICY_TRAINING`
- Policy examples: 1,060
- Source scenarios: 106
- Player-side examples: 332
- Opponent-side examples: 728
- Counterfactual examples: 212
- Validation checks passed: 21/21
- Search failures: 0
- Rejected policy records: 0

#### The corrected dataset contains real expert actions such as:

- `Ascension`
- `Bind Down`
- `Quick Attack`
- `Live Coal`
- `Pass`

#### `Pass` should occur only in legitimate states where no affordable attack is available.

## Code cell — Section 1A: imports and project paths

In [2]:
# ======================================================================================
# SECTION 1A — IMPORTS AND PROJECT PATHS
# ======================================================================================

from __future__ import annotations

import json
import math
import os
import random
import sys
import warnings

from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# --------------------------------------------------------------------------------------
# Reproducibility
# --------------------------------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)


# --------------------------------------------------------------------------------------
# Resolve project root
# --------------------------------------------------------------------------------------

CURRENT_DIRECTORY = Path.cwd().resolve()

if CURRENT_DIRECTORY.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    possible_root = CURRENT_DIRECTORY

    while (
        possible_root.parent != possible_root
        and not (possible_root / "notebooks").exists()
    ):
        possible_root = possible_root.parent

    PROJECT_ROOT = possible_root


NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

NOTEBOOK49_REPORT_DIR = REPORTS_DIR / "notebook49"
NOTEBOOK50_REPORT_DIR = REPORTS_DIR / "notebook50"

NOTEBOOK50_MODEL_DIR = MODELS_DIR / "notebook50"

NOTEBOOK50_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NOTEBOOK50_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("=" * 100)
print("SECTION 1A — IMPORTS AND PROJECT PATHS")
print("=" * 100)

print()
print("DIRECTORIES")
print("-" * 100)

print(f"Current directory       : {CURRENT_DIRECTORY}")
print(f"Project root            : {PROJECT_ROOT}")
print(f"Notebook 49 reports     : {NOTEBOOK49_REPORT_DIR}")
print(f"Notebook 50 reports     : {NOTEBOOK50_REPORT_DIR}")
print(f"Notebook 50 models      : {NOTEBOOK50_MODEL_DIR}")

print()
print("PACKAGE VERSIONS")
print("-" * 100)

print(f"Python                   : {sys.version.split()[0]}")
print(f"NumPy                    : {np.__version__}")
print(f"Pandas                   : {pd.__version__}")

print()
print("✅ SECTION 1A IMPORTS AND PROJECT PATHS PASSED")

SECTION 1A — IMPORTS AND PROJECT PATHS

DIRECTORIES
----------------------------------------------------------------------------------------------------
Current directory       : D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\notebooks
Project root            : D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge
Notebook 49 reports     : D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook49
Notebook 50 reports     : D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50
Notebook 50 models      : D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50

PACKAGE VERSIONS
----------------------------------------------------------------------------------------------------
Python                   : 3.13.3
NumPy                    : 2.2.6
Pandas                   : 2.2.3

✅ SECTION 1A IMPORTS AND PROJECT PATHS PASSED


## Section 1B — Notebook 49 Handoff Validation

#### This section validates the outputs produced by Notebook 49 before policy training begins.

##### The notebook verifies:

- final handoff manifest
- policy dataset
- policy statistics
- side distribution
- variant distribution
- search summary

#### Training is not allowed to continue unless every required artifact exists and the Notebook 49 status is:

#### READY_FOR_POLICY_TRAINING

In [4]:
# ======================================================================================
# SECTION 1B — NOTEBOOK 49 HANDOFF VALIDATION
# ======================================================================================

from pathlib import Path
import json

print("=" * 100)
print("SECTION 1B — NOTEBOOK 49 HANDOFF VALIDATION")
print("=" * 100)

FINAL_VALIDATION_DIR = (
    NOTEBOOK49_REPORT_DIR
    / "section8"
    / "final_validation"
)

POLICY_DATASET_DIR = (
    NOTEBOOK49_REPORT_DIR
    / "section8"
    / "policy_dataset"
)

required_files = {
    "handoff_manifest":
        FINAL_VALIDATION_DIR / "section8d_handoff_manifest.json",

    "final_summary":
        FINAL_VALIDATION_DIR / "section8d_final_summary.json",

    "policy_dataset_csv":
        POLICY_DATASET_DIR / "section8c_expert_policy_dataset.csv",

    "policy_statistics":
        POLICY_DATASET_DIR / "section8c_dataset_statistics.csv",

    "side_summary":
        POLICY_DATASET_DIR / "section8c_side_summary.csv",

    "variant_summary":
        POLICY_DATASET_DIR / "section8c_variant_summary.csv",
}

print()
print("REQUIRED ARTIFACTS")
print("-" * 100)

validation_rows = []

for name, path in required_files.items():

    exists = path.exists()

    size = path.stat().st_size if exists else 0

    validation_rows.append(
        {
            "artifact": name,
            "exists": exists,
            "size_bytes": size,
            "path": str(path),
        }
    )

validation_df = pd.DataFrame(validation_rows)

display(validation_df)

assert validation_df["exists"].all(), \
    "Notebook 49 artifacts are missing."

manifest_path = required_files["handoff_manifest"]

with open(
    manifest_path,
    "r",
    encoding="utf-8",
) as f:

    handoff_manifest = json.load(f)

print()
print("HANDOFF STATUS")
print("-" * 100)

for k, v in handoff_manifest.items():
    print(f"{k:30}: {v}")

assert (
    handoff_manifest["status"]
    == "READY_FOR_POLICY_TRAINING"
), "Notebook 49 was not completed successfully."

print()
print("✅ SECTION 1B NOTEBOOK 49 HANDOFF VALIDATION PASSED")

SECTION 1B — NOTEBOOK 49 HANDOFF VALIDATION

REQUIRED ARTIFACTS
----------------------------------------------------------------------------------------------------


,artifact,exists,size_bytes,path
0,handoff_manifest,True,2257,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook49\section8\final_validation\section8d_h...
1,final_summary,True,510,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook49\section8\final_validation\section8d_f...
2,policy_dataset_csv,True,2668658,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook49\section8\policy_dataset\section8c_exp...
3,policy_statistics,True,386,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook49\section8\policy_dataset\section8c_dat...
4,side_summary,True,325,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook49\section8\policy_dataset\section8c_sid...
5,variant_summary,True,1664,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook49\section8\policy_dataset\section8c_var...



HANDOFF STATUS
----------------------------------------------------------------------------------------------------
notebook                      : 49_side_balanced_search_guided_self_play
status                        : READY_FOR_POLICY_TRAINING
completed_sections            : ['7A', '7B', '7C', '8A', '8B', '8C', '8D']
final_dataset_rows            : 1060
source_scenarios              : 106
unique_matchups               : 4
player_side_examples          : 332
opponent_side_examples        : 728
opponent_side_fraction        : 0.6867924528301886
preserved_side_examples       : 848
counterfactual_side_examples  : 212
rejected_records              : 0
variant_failures              : 0
search_failures               : 0
search_pending                : 0
validation_checks_passed      : 21
validation_checks_failed      : 0
failed_checks                 : []
training_dataset_csv          : D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook49\section8\policy_dataset\

## Section 1C — Policy Dataset Loading and Class Inspection

#### This section loads the corrected expert-policy dataset from Notebook 49 and inspects:

- dataset shape
- column structure
- missing values
- expert move distribution
- side distribution
- legal-move counts
- value-target range
- sample-weight range

#### This inspection ensures that the policy labels are no longer dominated by `Pass`.

In [5]:
# ======================================================================================
# SECTION 1C — POLICY DATASET LOADING AND CLASS INSPECTION
# ======================================================================================

print("=" * 100)
print("SECTION 1C — POLICY DATASET LOADING AND CLASS INSPECTION")
print("=" * 100)

POLICY_DATASET_FILE = required_files["policy_dataset_csv"]

policy_df = pd.read_csv(
    POLICY_DATASET_FILE,
    low_memory=False,
)

assert not policy_df.empty, \
    "The Notebook 49 policy dataset is empty."

print()
print("DATASET SHAPE")
print("-" * 100)

print(f"Rows                     : {len(policy_df)}")
print(f"Columns                  : {policy_df.shape[1]}")

print()
print("COLUMN NAMES")
print("-" * 100)

for index, column in enumerate(policy_df.columns, start=1):
    print(f"{index:>3}. {column}")

required_policy_columns = [
    "variant_id",
    "example_id",
    "source_scenario_id",
    "variant_name",
    "current_side",
    "side_mode",
    "state_key",
    "legal_move_count",
    "expert_move",
    "value_target",
    "search_depth_reached",
    "sample_weight",
]

missing_columns = [
    column
    for column in required_policy_columns
    if column not in policy_df.columns
]

assert not missing_columns, (
    "Required policy columns are missing: "
    f"{missing_columns}"
)

print()
print("MISSING VALUES IN REQUIRED COLUMNS")
print("-" * 100)

required_missing_df = (
    policy_df[
        required_policy_columns
    ]
    .isna()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)

display(required_missing_df)

assert (
    required_missing_df["missing_count"] == 0
).all(), "Required policy columns contain missing values."

print()
print("EXPERT MOVE DISTRIBUTION")
print("-" * 100)

expert_move_distribution = (
    policy_df["expert_move"]
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("expert_move")
    .reset_index(name="examples")
)

expert_move_distribution["fraction"] = (
    expert_move_distribution["examples"]
    / len(policy_df)
)

display(expert_move_distribution)

print()
print("SIDE DISTRIBUTION")
print("-" * 100)

side_distribution = (
    policy_df["current_side"]
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("current_side")
    .reset_index(name="examples")
)

side_distribution["fraction"] = (
    side_distribution["examples"]
    / len(policy_df)
)

display(side_distribution)

print()
print("LEGAL MOVE COUNT DISTRIBUTION")
print("-" * 100)

legal_move_distribution = (
    pd.to_numeric(
        policy_df["legal_move_count"],
        errors="coerce",
    )
    .value_counts()
    .sort_index()
    .rename_axis("legal_move_count")
    .reset_index(name="examples")
)

display(legal_move_distribution)

print()
print("NUMERIC TARGET PROFILE")
print("-" * 100)

numeric_profile = pd.DataFrame(
    {
        "metric": [
            "value_target_min",
            "value_target_mean",
            "value_target_max",
            "sample_weight_min",
            "sample_weight_mean",
            "sample_weight_max",
            "search_depth_min",
            "search_depth_mean",
            "search_depth_max",
        ],
        "value": [
            pd.to_numeric(
                policy_df["value_target"],
                errors="coerce",
            ).min(),

            pd.to_numeric(
                policy_df["value_target"],
                errors="coerce",
            ).mean(),

            pd.to_numeric(
                policy_df["value_target"],
                errors="coerce",
            ).max(),

            pd.to_numeric(
                policy_df["sample_weight"],
                errors="coerce",
            ).min(),

            pd.to_numeric(
                policy_df["sample_weight"],
                errors="coerce",
            ).mean(),

            pd.to_numeric(
                policy_df["sample_weight"],
                errors="coerce",
            ).max(),

            pd.to_numeric(
                policy_df["search_depth_reached"],
                errors="coerce",
            ).min(),

            pd.to_numeric(
                policy_df["search_depth_reached"],
                errors="coerce",
            ).mean(),

            pd.to_numeric(
                policy_df["search_depth_reached"],
                errors="coerce",
            ).max(),
        ],
    }
)

display(numeric_profile)

policy_classes = sorted(
    policy_df["expert_move"]
    .astype(str)
    .unique()
    .tolist()
)

print()
print("POLICY CLASSES")
print("-" * 100)

print(f"Policy class count       : {len(policy_classes)}")
print(f"Policy classes           : {policy_classes}")

assert len(policy_classes) > 1, (
    "The policy dataset still contains only one action class."
)

assert not (
    len(policy_classes) == 1
    and policy_classes[0] == "Pass"
), "The policy dataset is still an all-Pass dataset."

print()
print("✅ SECTION 1C POLICY DATASET LOADING AND CLASS INSPECTION PASSED")

SECTION 1C — POLICY DATASET LOADING AND CLASS INSPECTION

DATASET SHAPE
----------------------------------------------------------------------------------------------------
Rows                     : 1060
Columns                  : 37

COLUMN NAMES
----------------------------------------------------------------------------------------------------
  1. example_id
  2. variant_id
  3. source_scenario_id
  4. source_match_id
  5. variant_name
  6. variant_seed
  7. queue_position
  8. curriculum_priority
  9. hard_example_rank
 10. side_mode
 11. source_side
 12. current_side
 13. baseline_name
 14. player_card
 15. opponent_card
 16. turn_number
 17. player_energy
 18. opponent_energy
 19. player_damage
 20. opponent_damage
 21. prize_cards_remaining
 22. hand_size
 23. state_key
 24. legal_move_count
 25. legal_moves
 26. expert_move
 27. expert_move_key
 28. value_target
 29. search_depth_requested
 30. search_depth_reached
 31. search_nodes
 32. search_elapsed_seconds
 33. search_tim

,column,missing_count
0,variant_id,0
1,example_id,0
2,source_scenario_id,0
3,variant_name,0
4,current_side,0
5,side_mode,0
6,state_key,0
7,legal_move_count,0
8,expert_move,0
9,value_target,0



EXPERT MOVE DISTRIBUTION
----------------------------------------------------------------------------------------------------


,expert_move,examples,fraction
0,Ascension,424,0.400000
1,Quick Attack,318,0.300000
2,Pass,125,0.117925
3,Bind Down,94,0.088679
4,Live Coal,80,0.075472
5,Tuck Tail,19,0.017925



SIDE DISTRIBUTION
----------------------------------------------------------------------------------------------------


...


LEGAL MOVE COUNT DISTRIBUTION
----------------------------------------------------------------------------------------------------


,legal_move_count,examples
0,1,742
1,2,318



NUMERIC TARGET PROFILE
----------------------------------------------------------------------------------------------------


,metric,value
0,value_target_min,-633.000000
1,value_target_mean,-1.177406
2,value_target_max,613.000000
3,sample_weight_min,0.990000
4,sample_weight_mean,1.523857
5,sample_weight_max,1.878500
6,search_depth_min,4.000000
7,search_depth_mean,4.000000
8,search_depth_max,4.000000



POLICY CLASSES
----------------------------------------------------------------------------------------------------
Policy class count       : 6
Policy classes           : ['Ascension', 'Bind Down', 'Live Coal', 'Pass', 'Quick Attack', 'Tuck Tail']

✅ SECTION 1C POLICY DATASET LOADING AND CLASS INSPECTION PASSED


# Section 2 — Side-Aware Dataset Preparation

#### This section prepares the policy-training dataset.

#### Goals:

- Build the feature matrix.
- Encode the target policy labels.
- Preserve scenario independence.
- Split the dataset into:
  - Training
  - Validation
  - Test
- Verify no scenario leakage.
- Report side balance within each split.

#### The split is performed at the source_scenario_id level rather than the individual example level.

In [6]:
# ======================================================================================
# SECTION 2A — FEATURE SELECTION
# ======================================================================================

print("=" * 100)
print("SECTION 2A — FEATURE SELECTION")
print("=" * 100)

FEATURE_COLUMNS = [

    "variant_name",
    "current_side",
    "side_mode",

    "player_card",
    "opponent_card",

    "turn_number",

    "player_energy",
    "opponent_energy",

    "player_damage",
    "opponent_damage",

    "prize_cards_remaining",

    "hand_size",

    "legal_move_count",

]

TARGET_COLUMN = "expert_move"

GROUP_COLUMN = "source_scenario_id"

WEIGHT_COLUMN = "sample_weight"

X = policy_df[FEATURE_COLUMNS].copy()

y = policy_df[TARGET_COLUMN].copy()

groups = policy_df[GROUP_COLUMN].copy()

sample_weights = (
    pd.to_numeric(
        policy_df[WEIGHT_COLUMN],
        errors="coerce",
    )
)

print()
print("FEATURE SUMMARY")
print("-" * 100)

print(f"Examples             : {len(X)}")
print(f"Feature columns      : {len(FEATURE_COLUMNS)}")
print(f"Policy classes       : {y.nunique()}")
print(f"Source scenarios     : {groups.nunique()}")

print()

display(X.head())

print()

display(y.head())

assert len(X) == len(y)

assert len(groups) == len(X)

assert sample_weights.notna().all()

print()
print("✅ SECTION 2A FEATURE SELECTION PASSED")

SECTION 2A — FEATURE SELECTION

FEATURE SUMMARY
----------------------------------------------------------------------------------------------------
Examples             : 1060
Feature columns      : 13
Policy classes       : 6
Source scenarios     : 106



,variant_name,current_side,side_mode,player_card,opponent_card,turn_number,player_energy,opponent_energy,player_damage,opponent_damage,prize_cards_remaining,hand_size,legal_move_count
0,balanced_midgame,Opponent,PRESERVE,Bulbasaur,Eevee,5,2,2,20.0,12.5,4,5,1
1,counterfactual_side_early,Player,FLIP,Bulbasaur,Eevee,3,1,1,8.0,5.0,5,6,1
2,counterfactual_side_late,Player,FLIP,Bulbasaur,Eevee,8,3,3,36.0,22.5,3,4,1
3,critical_hp_decision,Opponent,PRESERVE,Bulbasaur,Eevee,10,4,4,56.0,27.5,2,3,2
4,early_pressure,Opponent,PRESERVE,Bulbasaur,Eevee,3,1,2,12.0,2.5,5,6,1


0       Ascension
1       Bind Down
2       Bind Down
3    Quick Attack
4       Ascension
Name: expert_move, dtype: object


✅ SECTION 2A FEATURE SELECTION PASSED


## Section 2B — Encode Policy Labels.

In [7]:
# ======================================================================================
# SECTION 2B — LABEL ENCODING
# ======================================================================================

from sklearn.preprocessing import LabelEncoder

print("=" * 100)
print("SECTION 2B — LABEL ENCODING")
print("=" * 100)

policy_label_encoder = LabelEncoder()

y_encoded = policy_label_encoder.fit_transform(y)

policy_classes = list(
    policy_label_encoder.classes_
)

policy_lookup = pd.DataFrame(
    {
        "label_id": range(len(policy_classes)),
        "policy_move": policy_classes,
    }
)

print()

display(policy_lookup)

print()

print(f"Total policy classes : {len(policy_classes)}")

assert len(policy_classes) == y.nunique()

print()

print("✅ SECTION 2B LABEL ENCODING PASSED")

SECTION 2B — LABEL ENCODING



,label_id,policy_move
0,0,Ascension
1,1,Bind Down
2,2,Live Coal
3,3,Pass
4,4,Quick Attack
5,5,Tuck Tail



Total policy classes : 6

✅ SECTION 2B LABEL ENCODING PASSED


## Section 2C — Group-Aware Train / Validation / Test Split

In [8]:
# ======================================================================================
# SECTION 2C — GROUP-AWARE DATA SPLIT
# ======================================================================================

print("=" * 100)
print("SECTION 2C — GROUP-AWARE DATA SPLIT")
print("=" * 100)

gss_outer = GroupShuffleSplit(
    n_splits=1,
    train_size=0.80,
    random_state=RANDOM_SEED,
)

train_index, temp_index = next(
    gss_outer.split(
        X,
        y_encoded,
        groups,
    )
)

X_train = X.iloc[train_index].reset_index(drop=True)
y_train = y_encoded[train_index]

groups_train = groups.iloc[train_index].reset_index(drop=True)

weights_train = sample_weights.iloc[
    train_index
].reset_index(drop=True)

X_temp = X.iloc[temp_index].reset_index(drop=True)
y_temp = y_encoded[temp_index]

groups_temp = groups.iloc[temp_index].reset_index(drop=True)

weights_temp = sample_weights.iloc[
    temp_index
].reset_index(drop=True)

gss_inner = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=RANDOM_SEED,
)

valid_index, test_index = next(
    gss_inner.split(
        X_temp,
        y_temp,
        groups_temp,
    )
)

X_valid = X_temp.iloc[valid_index].reset_index(drop=True)
y_valid = y_temp[valid_index]

groups_valid = groups_temp.iloc[
    valid_index
].reset_index(drop=True)

weights_valid = weights_temp.iloc[
    valid_index
].reset_index(drop=True)

X_test = X_temp.iloc[test_index].reset_index(drop=True)
y_test = y_temp[test_index]

groups_test = groups_temp.iloc[
    test_index
].reset_index(drop=True)

weights_test = weights_temp.iloc[
    test_index
].reset_index(drop=True)

print()

print("SPLIT SIZES")
print("-" * 100)

print(f"Training examples      : {len(X_train)}")
print(f"Validation examples    : {len(X_valid)}")
print(f"Test examples          : {len(X_test)}")

print()

print(f"Training scenarios     : {groups_train.nunique()}")
print(f"Validation scenarios   : {groups_valid.nunique()}")
print(f"Test scenarios         : {groups_test.nunique()}")

assert (
    set(groups_train)
    & set(groups_valid)
) == set()

assert (
    set(groups_train)
    & set(groups_test)
) == set()

assert (
    set(groups_valid)
    & set(groups_test)
) == set()

print()

print("✅ SECTION 2C GROUP SPLITTING PASSED")

SECTION 2C — GROUP-AWARE DATA SPLIT

SPLIT SIZES
----------------------------------------------------------------------------------------------------
Training examples      : 840
Validation examples    : 110
Test examples          : 110

Training scenarios     : 84
Validation scenarios   : 11
Test scenarios         : 11

✅ SECTION 2C GROUP SPLITTING PASSED


# Section 3 — Policy Feature Engineering and Baseline Model

## This section prepares the feature preprocessing pipeline.

#### Steps:

1. Separate categorical and numeric features.
2. Build preprocessing transformers.
3. Fit a baseline Random Forest policy model.
4. Train using Notebook 49 sample weights.
5. Export the preprocessing pipeline.

## Section 3A — Build the preprocessing pipeline

In [9]:
# ======================================================================================
# SECTION 3A — FEATURE PREPROCESSING PIPELINE
# ======================================================================================

print("=" * 100)
print("SECTION 3A — FEATURE PREPROCESSING PIPELINE")
print("=" * 100)

categorical_features = [

    "variant_name",

    "current_side",

    "side_mode",

    "player_card",

    "opponent_card",

]

numeric_features = [

    "turn_number",

    "player_energy",

    "opponent_energy",

    "player_damage",

    "opponent_damage",

    "prize_cards_remaining",

    "hand_size",

    "legal_move_count",

]

numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
    ]
)

print()

print("NUMERIC FEATURES")
print("-" * 100)

for feature in numeric_features:
    print(feature)

print()

print("CATEGORICAL FEATURES")
print("-" * 100)

for feature in categorical_features:
    print(feature)

print()

print(f"Numeric feature count      : {len(numeric_features)}")
print(f"Categorical feature count  : {len(categorical_features)}")

assert (
    len(numeric_features)
    + len(categorical_features)
    == len(FEATURE_COLUMNS)
)

print()

print("✅ SECTION 3A FEATURE PREPROCESSING PASSED")

SECTION 3A — FEATURE PREPROCESSING PIPELINE

NUMERIC FEATURES
----------------------------------------------------------------------------------------------------
turn_number
player_energy
opponent_energy
player_damage
opponent_damage
prize_cards_remaining
hand_size
legal_move_count

CATEGORICAL FEATURES
----------------------------------------------------------------------------------------------------
variant_name
current_side
side_mode
player_card
opponent_card

Numeric feature count      : 8
Categorical feature count  : 5

✅ SECTION 3A FEATURE PREPROCESSING PASSED


## Section 3B — Fit the preprocessor

In [10]:
# ======================================================================================
# SECTION 3B — FIT FEATURE PREPROCESSOR
# ======================================================================================

print("=" * 100)
print("SECTION 3B — FIT FEATURE PREPROCESSOR")
print("=" * 100)

X_train_processed = preprocessor.fit_transform(
    X_train
)

X_valid_processed = preprocessor.transform(
    X_valid
)

X_test_processed = preprocessor.transform(
    X_test
)

print()

print("FEATURE MATRIX SHAPES")
print("-" * 100)

print(f"Training      : {X_train_processed.shape}")
print(f"Validation    : {X_valid_processed.shape}")
print(f"Test          : {X_test_processed.shape}")

feature_names = preprocessor.get_feature_names_out()

print()

print(f"Expanded feature count : {len(feature_names)}")

assert (
    X_train_processed.shape[1]
    == len(feature_names)
)

print()

print("First 20 encoded features")

print("-" * 100)

for feature in feature_names[:20]:
    print(feature)

print()

print("✅ SECTION 3B FEATURE PREPROCESSOR PASSED")

SECTION 3B — FIT FEATURE PREPROCESSOR

FEATURE MATRIX SHAPES
----------------------------------------------------------------------------------------------------
Training      : (840, 28)
Validation    : (110, 28)
Test          : (110, 28)

Expanded feature count : 28

First 20 encoded features
----------------------------------------------------------------------------------------------------
numeric__turn_number
numeric__player_energy
numeric__opponent_energy
numeric__player_damage
numeric__opponent_damage
numeric__prize_cards_remaining
numeric__hand_size
numeric__legal_move_count
categorical__variant_name_balanced_midgame
categorical__variant_name_counterfactual_side_early
categorical__variant_name_counterfactual_side_late
categorical__variant_name_critical_hp_decision
categorical__variant_name_early_pressure
categorical__variant_name_late_game_prize_pressure
categorical__variant_name_opening_one_energy
categorical__variant_name_opening_zero_energy
categorical__variant_name_opponent

# Section 4 — Baseline Policy Model

#### This section trains the first policy network surrogate using the corrected expert policy dataset.

#### The baseline uses a Random Forest classifier because it:

- supports multiclass prediction
- accepts sample weights
- is fast to iterate
- provides feature importance
- establishes a strong baseline before neural policy training

## Section 4A — Train the Baseline Policy Model

In [11]:
# ======================================================================================
# SECTION 4A — TRAIN BASELINE RANDOM FOREST POLICY
# ======================================================================================

print("=" * 100)
print("SECTION 4A — BASELINE POLICY TRAINING")
print("=" * 100)

baseline_policy = RandomForestClassifier(

    n_estimators=500,

    max_depth=None,

    min_samples_leaf=2,

    random_state=RANDOM_SEED,

    n_jobs=-1,

)

baseline_policy.fit(

    X_train_processed,

    y_train,

    sample_weight=weights_train,

)

print()

print("MODEL")

print("-" * 100)

print(baseline_policy)

print()

print("Training complete.")

print()

print("✅ SECTION 4A BASELINE TRAINING PASSED")

SECTION 4A — BASELINE POLICY TRAINING

MODEL
----------------------------------------------------------------------------------------------------
RandomForestClassifier(min_samples_leaf=2, n_estimators=500, n_jobs=-1,
                       random_state=42)

Training complete.

✅ SECTION 4A BASELINE TRAINING PASSED


## Section 4B — Evaluate the Model

In [12]:
# ======================================================================================
# SECTION 4B — POLICY EVALUATION
# ======================================================================================

print("=" * 100)
print("SECTION 4B — POLICY EVALUATION")
print("=" * 100)

train_predictions = baseline_policy.predict(
    X_train_processed
)

valid_predictions = baseline_policy.predict(
    X_valid_processed
)

test_predictions = baseline_policy.predict(
    X_test_processed
)

evaluation_summary = pd.DataFrame({

    "Split":[
        "Training",
        "Validation",
        "Test",
    ],

    "Accuracy":[

        accuracy_score(
            y_train,
            train_predictions,
        ),

        accuracy_score(
            y_valid,
            valid_predictions,
        ),

        accuracy_score(
            y_test,
            test_predictions,
        ),

    ],

    "Balanced Accuracy":[

        balanced_accuracy_score(
            y_train,
            train_predictions,
        ),

        balanced_accuracy_score(
            y_valid,
            valid_predictions,
        ),

        balanced_accuracy_score(
            y_test,
            test_predictions,
        ),

    ],

    "Macro F1":[

        f1_score(
            y_train,
            train_predictions,
            average="macro",
        ),

        f1_score(
            y_valid,
            valid_predictions,
            average="macro",
        ),

        f1_score(
            y_test,
            test_predictions,
            average="macro",
        ),

    ],

})

display(evaluation_summary)

print()

print("Validation Classification Report")

print("-" * 100)

print(

    classification_report(

        y_valid,

        valid_predictions,

        target_names=policy_classes,

    )

)

print()

print("Test Classification Report")

print("-" * 100)

print(

    classification_report(

        y_test,

        test_predictions,

        target_names=policy_classes,

    )

)

print()

print("✅ SECTION 4B POLICY EVALUATION PASSED")

SECTION 4B — POLICY EVALUATION


,Split,Accuracy,Balanced Accuracy,Macro F1
0,Training,1.0,1.0,1.0
1,Validation,1.0,1.0,1.0
2,Test,1.0,1.0,1.0



Validation Classification Report
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

   Ascension       1.00      1.00      1.00        44
   Bind Down       1.00      1.00      1.00         2
   Live Coal       1.00      1.00      1.00        10
        Pass       1.00      1.00      1.00        16
Quick Attack       1.00      1.00      1.00        33
   Tuck Tail       1.00      1.00      1.00         5

    accuracy                           1.00       110
   macro avg       1.00      1.00      1.00       110
weighted avg       1.00      1.00      1.00       110


Test Classification Report
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

   Ascension       1.00      1.00      1.00        44
   Bind Down       1.00      1.00      1.00        12
   Live Coal       1.00      1

## Section 4C — Scenario-Label Leakage Check

#### The baseline model achieved perfect validation and test performance.

#### Because `variant_name` may strongly encode the intended decision type, this section retrains the policy model without that feature.

#### The goal is to determine whether the model is learning from actual battle-state variables or memorizing scenario labels.

In [13]:
# ======================================================================================
# SECTION 4C — RETRAIN WITHOUT VARIANT_NAME
# ======================================================================================

print("=" * 100)
print("SECTION 4C — SCENARIO-LABEL LEAKAGE CHECK")
print("=" * 100)

LEAKAGE_SAFE_FEATURE_COLUMNS = [
    column
    for column in FEATURE_COLUMNS
    if column != "variant_name"
]

leakage_safe_categorical_features = [
    "current_side",
    "side_mode",
    "player_card",
    "opponent_card",
]

leakage_safe_numeric_features = [
    "turn_number",
    "player_energy",
    "opponent_energy",
    "player_damage",
    "opponent_damage",
    "prize_cards_remaining",
    "hand_size",
    "legal_move_count",
]

leakage_safe_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="median"),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

leakage_safe_categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

leakage_safe_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            leakage_safe_numeric_pipeline,
            leakage_safe_numeric_features,
        ),
        (
            "categorical",
            leakage_safe_categorical_pipeline,
            leakage_safe_categorical_features,
        ),
    ]
)

X_train_safe = X_train[
    LEAKAGE_SAFE_FEATURE_COLUMNS
].copy()

X_valid_safe = X_valid[
    LEAKAGE_SAFE_FEATURE_COLUMNS
].copy()

X_test_safe = X_test[
    LEAKAGE_SAFE_FEATURE_COLUMNS
].copy()

X_train_safe_processed = (
    leakage_safe_preprocessor.fit_transform(
        X_train_safe
    )
)

X_valid_safe_processed = (
    leakage_safe_preprocessor.transform(
        X_valid_safe
    )
)

X_test_safe_processed = (
    leakage_safe_preprocessor.transform(
        X_test_safe
    )
)

leakage_safe_policy = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

leakage_safe_policy.fit(
    X_train_safe_processed,
    y_train,
    sample_weight=weights_train,
)

safe_train_predictions = leakage_safe_policy.predict(
    X_train_safe_processed
)

safe_valid_predictions = leakage_safe_policy.predict(
    X_valid_safe_processed
)

safe_test_predictions = leakage_safe_policy.predict(
    X_test_safe_processed
)

leakage_check_summary = pd.DataFrame(
    {
        "split": [
            "Training",
            "Validation",
            "Test",
        ],
        "baseline_accuracy": [
            accuracy_score(
                y_train,
                train_predictions,
            ),
            accuracy_score(
                y_valid,
                valid_predictions,
            ),
            accuracy_score(
                y_test,
                test_predictions,
            ),
        ],
        "without_variant_name_accuracy": [
            accuracy_score(
                y_train,
                safe_train_predictions,
            ),
            accuracy_score(
                y_valid,
                safe_valid_predictions,
            ),
            accuracy_score(
                y_test,
                safe_test_predictions,
            ),
        ],
        "without_variant_name_balanced_accuracy": [
            balanced_accuracy_score(
                y_train,
                safe_train_predictions,
            ),
            balanced_accuracy_score(
                y_valid,
                safe_valid_predictions,
            ),
            balanced_accuracy_score(
                y_test,
                safe_test_predictions,
            ),
        ],
        "without_variant_name_macro_f1": [
            f1_score(
                y_train,
                safe_train_predictions,
                average="macro",
            ),
            f1_score(
                y_valid,
                safe_valid_predictions,
                average="macro",
            ),
            f1_score(
                y_test,
                safe_test_predictions,
                average="macro",
            ),
        ],
    }
)

print()
print("LEAKAGE CHECK RESULTS")
print("-" * 100)

display(leakage_check_summary)

print()
print("VALIDATION REPORT WITHOUT VARIANT_NAME")
print("-" * 100)

print(
    classification_report(
        y_valid,
        safe_valid_predictions,
        labels=np.arange(
            len(policy_classes)
        ),
        target_names=policy_classes,
        zero_division=0,
    )
)

print()
print("TEST REPORT WITHOUT VARIANT_NAME")
print("-" * 100)

print(
    classification_report(
        y_test,
        safe_test_predictions,
        labels=np.arange(
            len(policy_classes)
        ),
        target_names=policy_classes,
        zero_division=0,
    )
)

assert (
    X_train_safe_processed.shape[1]
    < X_train_processed.shape[1]
), "variant_name does not appear to have been removed."

print()
print("✅ SECTION 4C SCENARIO-LABEL LEAKAGE CHECK PASSED")

SECTION 4C — SCENARIO-LABEL LEAKAGE CHECK

LEAKAGE CHECK RESULTS
----------------------------------------------------------------------------------------------------


,split,baseline_accuracy,without_variant_name_accuracy,without_variant_name_balanced_accuracy,without_variant_name_macro_f1
0,Training,1.0,1.0,1.0,1.0
1,Validation,1.0,1.0,1.0,1.0
2,Test,1.0,1.0,1.0,1.0



VALIDATION REPORT WITHOUT VARIANT_NAME
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

   Ascension       1.00      1.00      1.00        44
   Bind Down       1.00      1.00      1.00         2
   Live Coal       1.00      1.00      1.00        10
        Pass       1.00      1.00      1.00        16
Quick Attack       1.00      1.00      1.00        33
   Tuck Tail       1.00      1.00      1.00         5

    accuracy                           1.00       110
   macro avg       1.00      1.00      1.00       110
weighted avg       1.00      1.00      1.00       110


TEST REPORT WITHOUT VARIANT_NAME
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

   Ascension       1.00      1.00      1.00        44
   Bind Down       1.00      1.00      1.00        12
   Live Coal      

## Section 4D — Leakage-Safe Feature Importance

#### This section identifies which encoded features the leakage-safe policy uses most heavily.

#### A reliable policy should emphasize battle-state variables such as energy, damage, legal-move count, side, and card identity rather than scenario labels.

In [14]:
# ======================================================================================
# SECTION 4D — LEAKAGE-SAFE FEATURE IMPORTANCE
# ======================================================================================

print("=" * 100)
print("SECTION 4D — LEAKAGE-SAFE FEATURE IMPORTANCE")
print("=" * 100)

safe_feature_names = (
    leakage_safe_preprocessor
    .get_feature_names_out()
)

safe_feature_importance_df = (
    pd.DataFrame(
        {
            "feature": safe_feature_names,
            "importance":
                leakage_safe_policy.feature_importances_,
        }
    )
    .sort_values(
        "importance",
        ascending=False,
    )
    .reset_index(drop=True)
)

safe_feature_importance_df[
    "cumulative_importance"
] = (
    safe_feature_importance_df[
        "importance"
    ].cumsum()
)

print()
print("TOP 20 FEATURES")
print("-" * 100)

display(
    safe_feature_importance_df.head(20)
)

print()
print(
    f"Total importance: "
    f"{safe_feature_importance_df['importance'].sum():.6f}"
)

assert np.isclose(
    safe_feature_importance_df[
        "importance"
    ].sum(),
    1.0,
)

print()
print("✅ SECTION 4D FEATURE IMPORTANCE PASSED")

SECTION 4D — LEAKAGE-SAFE FEATURE IMPORTANCE

TOP 20 FEATURES
----------------------------------------------------------------------------------------------------


,feature,importance,cumulative_importance
0,numeric__legal_move_count,0.181041,0.181041
1,numeric__opponent_energy,0.133203,0.314245
2,numeric__hand_size,0.116250,0.430495
3,numeric__turn_number,0.089311,0.519806
4,numeric__player_damage,0.075854,0.595660
5,numeric__player_energy,0.066596,0.662257
6,categorical__side_mode_FLIP,0.063865,0.726122
7,categorical__side_mode_PRESERVE,0.061696,0.787818
8,numeric__prize_cards_remaining,0.043735,0.831554
9,categorical__player_card_Bulbasaur,0.041970,0.873524



Total importance: 1.000000

✅ SECTION 4D FEATURE IMPORTANCE PASSED


# Section 4E — Decision-Only Evaluation

#### Overall policy accuracy is inflated by positions with only one legal move.

#### This section evaluates the model only on positions where the active player has two or more legal actions available.

#### These are the positions where policy quality actually matters.

In [15]:
# ======================================================================================
# SECTION 4E — DECISION-ONLY POLICY EVALUATION
# ======================================================================================

print("=" * 100)
print("SECTION 4E — DECISION-ONLY POLICY EVALUATION")
print("=" * 100)

valid_decision_mask = (
    X_valid["legal_move_count"] >= 2
)

test_decision_mask = (
    X_test["legal_move_count"] >= 2
)

X_valid_decision = X_valid_processed[
    valid_decision_mask.values
]

X_test_decision = X_test_processed[
    test_decision_mask.values
]

y_valid_decision = y_valid[
    valid_decision_mask.values
]

y_test_decision = y_test[
    test_decision_mask.values
]

valid_decision_predictions = baseline_policy.predict(
    X_valid_decision
)

test_decision_predictions = baseline_policy.predict(
    X_test_decision
)

decision_summary = pd.DataFrame({

    "Split":[
        "Validation",
        "Test",
    ],

    "Decision States":[
        len(y_valid_decision),
        len(y_test_decision),
    ],

    "Accuracy":[

        accuracy_score(
            y_valid_decision,
            valid_decision_predictions,
        ),

        accuracy_score(
            y_test_decision,
            test_decision_predictions,
        ),

    ],

    "Balanced Accuracy":[

        balanced_accuracy_score(
            y_valid_decision,
            valid_decision_predictions,
        ),

        balanced_accuracy_score(
            y_test_decision,
            test_decision_predictions,
        ),

    ],

    "Macro F1":[

        f1_score(
            y_valid_decision,
            valid_decision_predictions,
            average="macro",
        ),

        f1_score(
            y_test_decision,
            test_decision_predictions,
            average="macro",
        ),

    ],

})

print()

print("DECISION-ONLY SUMMARY")
print("-" * 100)

display(decision_summary)

print()

print("VALIDATION DECISION REPORT")
print("-" * 100)

print(

    classification_report(

        y_valid_decision,

        valid_decision_predictions,

        labels=np.arange(
            len(policy_classes)
        ),

        target_names=policy_classes,

        zero_division=0,

    )

)

print()

print("TEST DECISION REPORT")
print("-" * 100)

print(

    classification_report(

        y_test_decision,

        test_decision_predictions,

        labels=np.arange(
            len(policy_classes)
        ),

        target_names=policy_classes,

        zero_division=0,

    )

)

assert len(y_valid_decision) > 0
assert len(y_test_decision) > 0

print()
print("✅ SECTION 4E DECISION-ONLY EVALUATION PASSED")

SECTION 4E — DECISION-ONLY POLICY EVALUATION

DECISION-ONLY SUMMARY
----------------------------------------------------------------------------------------------------


C:\Users\johnb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
C:\Users\johnb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


,Split,Decision States,Accuracy,Balanced Accuracy,Macro F1
0,Validation,33,1.0,1.0,1.0
1,Test,33,1.0,1.0,1.0



VALIDATION DECISION REPORT
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

   Ascension       0.00      0.00      0.00         0
   Bind Down       0.00      0.00      0.00         0
   Live Coal       0.00      0.00      0.00         0
        Pass       0.00      0.00      0.00         0
Quick Attack       1.00      1.00      1.00        33
   Tuck Tail       0.00      0.00      0.00         0

    accuracy                           1.00        33
   macro avg       0.17      0.17      0.17        33
weighted avg       1.00      1.00      1.00        33


TEST DECISION REPORT
----------------------------------------------------------------------------------------------------
              precision    recall  f1-score   support

   Ascension       0.00      0.00      0.00         0
   Bind Down       0.00      0.00      0.00         0
   Live Coal       0.00      0.00      0.0

# Section 5 — Side-Balanced Policy Training

## The training dataset contains more Opponent-side examples than Player-side examples.

####This section:

1. Measures side imbalance in the training split.
2. Creates inverse-frequency side weights.
3. Combines side weights with the original Notebook 49 sample weights.
4. Trains a final leakage-safe, side-balanced Random Forest policy.
5. Compares it against the earlier baseline model.

## Section 5A — Construct Side-Balanced Training Weights

In [18]:
# ======================================================================================
# SECTION 5A — SIDE-BALANCED TRAINING WEIGHTS
# ======================================================================================

print("=" * 100)
print("SECTION 5A — SIDE-BALANCED TRAINING WEIGHTS")
print("=" * 100)

train_side_series = (
    X_train["current_side"]
    .astype(str)
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------
# Original Notebook 49 sample weights
# --------------------------------------------------------------------------------------

original_weights_train = (
    pd.to_numeric(
        weights_train,
        errors="coerce",
    )
    .fillna(1.0)
    .astype(float)
    .reset_index(drop=True)
)

# --------------------------------------------------------------------------------------
# Build side-level weighting table
#
# Balance the TOTAL ORIGINAL SAMPLE WEIGHT assigned to each side,
# not merely the number of rows on each side.
# --------------------------------------------------------------------------------------

side_weight_source_df = pd.DataFrame(
    {
        "current_side": train_side_series,
        "original_weight": original_weights_train,
    }
)

side_weight_summary_df = (
    side_weight_source_df
    .groupby(
        "current_side",
        as_index=False,
    )
    .agg(
        examples=("current_side", "size"),
        original_weight_total=("original_weight", "sum"),
        original_weight_mean=("original_weight", "mean"),
    )
)

side_weight_summary_df["fraction"] = (
    side_weight_summary_df["examples"]
    / len(side_weight_source_df)
)

number_of_sides = (
    side_weight_summary_df[
        "current_side"
    ].nunique()
)

total_original_weight = (
    side_weight_summary_df[
        "original_weight_total"
    ].sum()
)

target_weight_per_side = (
    total_original_weight
    / number_of_sides
)

side_weight_summary_df[
    "side_balance_factor"
] = (
    target_weight_per_side
    / side_weight_summary_df[
        "original_weight_total"
    ]
)

side_weight_lookup = dict(
    zip(
        side_weight_summary_df[
            "current_side"
        ],
        side_weight_summary_df[
            "side_balance_factor"
        ],
    )
)

side_balance_weights_train = (
    train_side_series
    .map(side_weight_lookup)
    .astype(float)
)

combined_weights_train = (
    original_weights_train
    * side_balance_weights_train
)

# Normalize the full training-weight vector to mean 1.0.
# This preserves the equal side totals.
combined_weights_train = (
    combined_weights_train
    / combined_weights_train.mean()
)

side_count_df = (
    side_weight_summary_df
    .rename(
        columns={
            "side_balance_factor":
                "inverse_frequency_weight",
        }
    )
    [
        [
            "current_side",
            "examples",
            "fraction",
            "original_weight_total",
            "original_weight_mean",
            "inverse_frequency_weight",
        ]
    ]
)

weight_profile_df = pd.DataFrame(
    {
        "metric": [
            "original_weight_min",
            "original_weight_mean",
            "original_weight_max",
            "side_weight_min",
            "side_weight_mean",
            "side_weight_max",
            "combined_weight_min",
            "combined_weight_mean",
            "combined_weight_max",
        ],
        "value": [
            original_weights_train.min(),
            original_weights_train.mean(),
            original_weights_train.max(),
            side_balance_weights_train.min(),
            side_balance_weights_train.mean(),
            side_balance_weights_train.max(),
            combined_weights_train.min(),
            combined_weights_train.mean(),
            combined_weights_train.max(),
        ],
    }
)

print()
print("WEIGHT PROFILE")
print("-" * 100)

display(weight_profile_df)

print()
print("WEIGHTED SIDE TOTALS")
print("-" * 100)

weighted_side_totals_df = (
    pd.DataFrame(
        {
            "current_side": train_side_series,
            "combined_weight": combined_weights_train,
        }
    )
    .groupby(
        "current_side",
        as_index=False,
    )
    .agg(
        examples=("combined_weight", "size"),
        total_combined_weight=("combined_weight", "sum"),
        average_combined_weight=("combined_weight", "mean"),
    )
)

display(weighted_side_totals_df)

assert side_balance_weights_train.notna().all()
assert combined_weights_train.notna().all()
assert np.isfinite(combined_weights_train).all()
assert (combined_weights_train > 0).all()
assert np.isclose(
    combined_weights_train.mean(),
    1.0,
)

weighted_side_totals = (
    weighted_side_totals_df[
        "total_combined_weight"
    ]
    .to_numpy()
)

relative_weight_gap = (
    weighted_side_totals.max()
    - weighted_side_totals.min()
) / weighted_side_totals.mean()

print()
print(f"Relative weighted-side gap : {relative_weight_gap:.4%}")

assert relative_weight_gap < 0.001, (
    "Combined weighting remains too imbalanced across sides."
)

print()
print("✅ SECTION 5A SIDE-BALANCED TRAINING WEIGHTS PASSED")

SECTION 5A — SIDE-BALANCED TRAINING WEIGHTS

WEIGHT PROFILE
----------------------------------------------------------------------------------------------------


,metric,value
0,original_weight_min,0.990000
1,original_weight_mean,1.519903
2,original_weight_max,1.878500
3,side_weight_min,0.657938
4,side_weight_mean,1.115961
5,side_weight_max,2.082898
6,combined_weight_min,0.578546
7,combined_weight_mean,1.000000
8,combined_weight_max,1.906933



WEIGHTED SIDE TOTALS
----------------------------------------------------------------------------------------------------


,current_side,examples,total_combined_weight,average_combined_weight
0,Opponent,570,420.0,0.736842
1,Player,270,420.0,1.555556



Relative weighted-side gap : 0.0000%

✅ SECTION 5A SIDE-BALANCED TRAINING WEIGHTS PASSED


## Section 5B — Final Side-Balanced Policy Model

#### This section trains the final Notebook 50 policy using:

- the leakage-safe feature set,
- original expert sample weights,
- inverse-frequency side-balancing weights.

#### The model is evaluated against the earlier leakage-safe baseline.

# ======================================================================================
# SECTION 5B — FINAL SIDE-BALANCED POLICY TRAINING
# ======================================================================================

print("=" * 100)
print("SECTION 5B — FINAL SIDE-BALANCED POLICY TRAINING")
print("=" * 100)

final_side_balanced_policy = RandomForestClassifier(
    n_estimators=750,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

final_side_balanced_policy.fit(
    X_train_safe_processed,
    y_train,
    sample_weight=combined_weights_train,
)

final_train_predictions = (
    final_side_balanced_policy.predict(
        X_train_safe_processed
    )
)

final_valid_predictions = (
    final_side_balanced_policy.predict(
        X_valid_safe_processed
    )
)

final_test_predictions = (
    final_side_balanced_policy.predict(
        X_test_safe_processed
    )
)

model_comparison_df = pd.DataFrame(
    {
        "model": [
            "Leakage-safe baseline",
            "Final side-balanced",
            "Leakage-safe baseline",
            "Final side-balanced",
            "Leakage-safe baseline",
            "Final side-balanced",
        ],
        "split": [
            "Training",
            "Training",
            "Validation",
            "Validation",
            "Test",
            "Test",
        ],
        "accuracy": [
            accuracy_score(
                y_train,
                safe_train_predictions,
            ),
            accuracy_score(
                y_train,
                final_train_predictions,
            ),
            accuracy_score(
                y_valid,
                safe_valid_predictions,
            ),
            accuracy_score(
                y_valid,
                final_valid_predictions,
            ),
            accuracy_score(
                y_test,
                safe_test_predictions,
            ),
            accuracy_score(
                y_test,
                final_test_predictions,
            ),
        ],
        "balanced_accuracy": [
            balanced_accuracy_score(
                y_train,
                safe_train_predictions,
            ),
            balanced_accuracy_score(
                y_train,
                final_train_predictions,
            ),
            balanced_accuracy_score(
                y_valid,
                safe_valid_predictions,
            ),
            balanced_accuracy_score(
                y_valid,
                final_valid_predictions,
            ),
            balanced_accuracy_score(
                y_test,
                safe_test_predictions,
            ),
            balanced_accuracy_score(
                y_test,
                final_test_predictions,
            ),
        ],
        "macro_f1": [
            f1_score(
                y_train,
                safe_train_predictions,
                average="macro",
                zero_division=0,
            ),
            f1_score(
                y_train,
                final_train_predictions,
                average="macro",
                zero_division=0,
            ),
            f1_score(
                y_valid,
                safe_valid_predictions,
                average="macro",
                zero_division=0,
            ),
            f1_score(
                y_valid,
                final_valid_predictions,
                average="macro",
                zero_division=0,
            ),
            f1_score(
                y_test,
                safe_test_predictions,
                average="macro",
                zero_division=0,
            ),
            f1_score(
                y_test,
                final_test_predictions,
                average="macro",
                zero_division=0,
            ),
        ],
    }
)

print()
print("MODEL COMPARISON")
print("-" * 100)

display(model_comparison_df)

print()
print("FINAL VALIDATION CLASSIFICATION REPORT")
print("-" * 100)

print(
    classification_report(
        y_valid,
        final_valid_predictions,
        labels=np.arange(
            len(policy_classes)
        ),
        target_names=policy_classes,
        zero_division=0,
    )
)

print()
print("FINAL TEST CLASSIFICATION REPORT")
print("-" * 100)

print(
    classification_report(
        y_test,
        final_test_predictions,
        labels=np.arange(
            len(policy_classes)
        ),
        target_names=policy_classes,
        zero_division=0,
    )
)

assert len(final_train_predictions) == len(y_train)
assert len(final_valid_predictions) == len(y_valid)
assert len(final_test_predictions) == len(y_test)

assert set(
    np.unique(
        final_train_predictions
    )
).issubset(
    set(
        range(
            len(policy_classes)
        )
    )
)

print()
print("✅ SECTION 5B FINAL SIDE-BALANCED POLICY TRAINING PASSED")

## Section 5C — Save Section 4 and Section 5 Reports

In [23]:
# ======================================================================================
# SECTION 5C — SAVE POLICY TRAINING AND EVALUATION REPORTS
# ======================================================================================

print("=" * 100)
print("SECTION 5C — SAVE POLICY TRAINING AND EVALUATION REPORTS")
print("=" * 100)

# --------------------------------------------------------------------------------------
# Report directories
# --------------------------------------------------------------------------------------

SECTION4_POLICY_TRAINING_DIR = (
    NOTEBOOK50_REPORT_DIR
    / "section4"
    / "policy_training"
)

SECTION5_POLICY_EVALUATION_DIR = (
    NOTEBOOK50_REPORT_DIR
    / "section5"
    / "policy_evaluation"
)

SECTION4_POLICY_TRAINING_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SECTION5_POLICY_EVALUATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# --------------------------------------------------------------------------------------
# Section 4 report paths
# --------------------------------------------------------------------------------------

SECTION4_BASELINE_EVALUATION_FILE = (
    SECTION4_POLICY_TRAINING_DIR
    / "section4b_baseline_evaluation.csv"
)

SECTION4_LEAKAGE_CHECK_FILE = (
    SECTION4_POLICY_TRAINING_DIR
    / "section4c_leakage_check.csv"
)

SECTION4_FEATURE_IMPORTANCE_FILE = (
    SECTION4_POLICY_TRAINING_DIR
    / "section4d_feature_importance.csv"
)

SECTION4_DECISION_EVALUATION_FILE = (
    SECTION4_POLICY_TRAINING_DIR
    / "section4e_decision_only_evaluation.csv"
)


# --------------------------------------------------------------------------------------
# Section 5 report paths
# --------------------------------------------------------------------------------------

SECTION5_SIDE_DISTRIBUTION_FILE = (
    SECTION5_POLICY_EVALUATION_DIR
    / "section5a_training_side_distribution.csv"
)

SECTION5_WEIGHT_PROFILE_FILE = (
    SECTION5_POLICY_EVALUATION_DIR
    / "section5a_weight_profile.csv"
)

SECTION5_WEIGHTED_SIDE_TOTALS_FILE = (
    SECTION5_POLICY_EVALUATION_DIR
    / "section5a_weighted_side_totals.csv"
)

SECTION5_MODEL_COMPARISON_FILE = (
    SECTION5_POLICY_EVALUATION_DIR
    / "section5b_model_comparison.csv"
)

SECTION5_SUMMARY_FILE = (
    SECTION5_POLICY_EVALUATION_DIR
    / "section5c_policy_evaluation_summary.json"
)


# --------------------------------------------------------------------------------------
# Save Section 4 reports
# --------------------------------------------------------------------------------------

evaluation_summary.to_csv(
    SECTION4_BASELINE_EVALUATION_FILE,
    index=False,
)

leakage_check_summary.to_csv(
    SECTION4_LEAKAGE_CHECK_FILE,
    index=False,
)

safe_feature_importance_df.to_csv(
    SECTION4_FEATURE_IMPORTANCE_FILE,
    index=False,
)

decision_summary.to_csv(
    SECTION4_DECISION_EVALUATION_FILE,
    index=False,
)


# --------------------------------------------------------------------------------------
# Save Section 5 reports
# --------------------------------------------------------------------------------------

side_count_df.to_csv(
    SECTION5_SIDE_DISTRIBUTION_FILE,
    index=False,
)

weight_profile_df.to_csv(
    SECTION5_WEIGHT_PROFILE_FILE,
    index=False,
)

weighted_side_totals_df.to_csv(
    SECTION5_WEIGHTED_SIDE_TOTALS_FILE,
    index=False,
)

model_comparison_df.to_csv(
    SECTION5_MODEL_COMPARISON_FILE,
    index=False,
)


# --------------------------------------------------------------------------------------
# Save Section 5 summary
# --------------------------------------------------------------------------------------

section5_summary = {
    "training_examples": int(len(X_train_safe)),
    "player_training_examples": int(
        (train_side_series == "Player").sum()
    ),
    "opponent_training_examples": int(
        (train_side_series == "Opponent").sum()
    ),
    "side_balancing_enabled": True,
    "opponent_total_combined_weight": float(
        weighted_side_totals_df.loc[
            weighted_side_totals_df["current_side"]
            == "Opponent",
            "total_combined_weight",
        ].iloc[0]
    ),
    "player_total_combined_weight": float(
        weighted_side_totals_df.loc[
            weighted_side_totals_df["current_side"]
            == "Player",
            "total_combined_weight",
        ].iloc[0]
    ),
    "relative_weighted_side_gap": float(
        relative_weight_gap
    ),
    "validation_accuracy": float(
        accuracy_score(
            y_valid,
            final_valid_predictions,
        )
    ),
    "validation_balanced_accuracy": float(
        balanced_accuracy_score(
            y_valid,
            final_valid_predictions,
        )
    ),
    "validation_macro_f1": float(
        f1_score(
            y_valid,
            final_valid_predictions,
            average="macro",
            zero_division=0,
        )
    ),
    "test_accuracy": float(
        accuracy_score(
            y_test,
            final_test_predictions,
        )
    ),
    "test_balanced_accuracy": float(
        balanced_accuracy_score(
            y_test,
            final_test_predictions,
        )
    ),
    "test_macro_f1": float(
        f1_score(
            y_test,
            final_test_predictions,
            average="macro",
            zero_division=0,
        )
    ),
    "next_stage": "MODEL_ARTIFACT_EXPORT",
}

with open(
    SECTION5_SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        section5_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------
# Validate saved reports
# --------------------------------------------------------------------------------------

section45_report_files = [
    SECTION4_BASELINE_EVALUATION_FILE,
    SECTION4_LEAKAGE_CHECK_FILE,
    SECTION4_FEATURE_IMPORTANCE_FILE,
    SECTION4_DECISION_EVALUATION_FILE,
    SECTION5_SIDE_DISTRIBUTION_FILE,
    SECTION5_WEIGHT_PROFILE_FILE,
    SECTION5_WEIGHTED_SIDE_TOTALS_FILE,
    SECTION5_MODEL_COMPARISON_FILE,
    SECTION5_SUMMARY_FILE,
]

section45_inventory_df = pd.DataFrame(
    [
        {
            "file": path.name,
            "path": str(path),
            "exists": path.exists(),
            "size_bytes": (
                path.stat().st_size
                if path.exists()
                else 0
            ),
            "nonempty": (
                path.exists()
                and path.stat().st_size > 0
            ),
        }
        for path in section45_report_files
    ]
)

print()
print("SAVED SECTION 4 AND SECTION 5 REPORTS")
print("-" * 100)

display(section45_inventory_df)

assert section45_inventory_df["exists"].all()
assert section45_inventory_df["nonempty"].all()

print()
print("SECTION 4 POLICY TRAINING REPORTS")
print("-" * 100)

print(SECTION4_BASELINE_EVALUATION_FILE)
print(SECTION4_LEAKAGE_CHECK_FILE)
print(SECTION4_FEATURE_IMPORTANCE_FILE)
print(SECTION4_DECISION_EVALUATION_FILE)

print()
print("SECTION 5 POLICY EVALUATION REPORTS")
print("-" * 100)

print(SECTION5_SIDE_DISTRIBUTION_FILE)
print(SECTION5_WEIGHT_PROFILE_FILE)
print(SECTION5_WEIGHTED_SIDE_TOTALS_FILE)
print(SECTION5_MODEL_COMPARISON_FILE)
print(SECTION5_SUMMARY_FILE)

print()
print("✅ SECTION 5C POLICY TRAINING AND EVALUATION REPORT EXPORT PASSED")

SECTION 5C — SAVE POLICY TRAINING AND EVALUATION REPORTS

SAVED SECTION 4 AND SECTION 5 REPORTS
----------------------------------------------------------------------------------------------------


,file,path,exists,size_bytes,nonempty
0,section4b_baseline_evaluation.csv,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section4\policy_training\section4b_ba...,True,107,True
1,section4c_leakage_check.csv,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section4\policy_training\section4c_le...,True,200,True
2,section4d_feature_importance.csv,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section4\policy_training\section4d_fe...,True,1295,True
3,section4e_decision_only_evaluation.csv,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section4\policy_training\section4e_de...,True,107,True
4,section5a_training_side_distribution.csv,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section5\policy_evaluation\section5a_...,True,256,True
5,section5a_weight_profile.csv,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section5\policy_evaluation\section5a_...,True,320,True
6,section5a_weighted_side_totals.csv,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section5\policy_evaluation\section5a_...,True,170,True
7,section5b_model_comparison.csv,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section5\policy_evaluation\section5b_...,True,303,True
8,section5c_policy_evaluation_summary.json,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section5\policy_evaluation\section5c_...,True,538,True



SECTION 4 POLICY TRAINING REPORTS
----------------------------------------------------------------------------------------------------
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section4\policy_training\section4b_baseline_evaluation.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section4\policy_training\section4c_leakage_check.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section4\policy_training\section4d_feature_importance.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section4\policy_training\section4e_decision_only_evaluation.csv

SECTION 5 POLICY EVALUATION REPORTS
----------------------------------------------------------------------------------------------------
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section5\policy_evaluation\section5a_training_side_distribution.csv
D:\02_AI_and_Data\Kaggle-AI-A

# Section 6 — Model Export and Reload Validation

## This section packages the final side-balanced policy for reuse by later notebooks and the simulator.

#### Artifacts exported:

- final side-balanced policy model
- leakage-safe preprocessing pipeline
- policy label encoder
- encoded feature names
- model configuration
- training and evaluation summaries
- side-balancing metadata

#### The exported artifacts are then reloaded and tested to confirm that deployment predictions match the in-memory model.

## Section 6A — Export Model Artifacts

In [20]:
# ======================================================================================
# SECTION 6A — EXPORT FINAL POLICY ARTIFACTS
# ======================================================================================

print("=" * 100)
print("SECTION 6A — EXPORT FINAL POLICY ARTIFACTS")
print("=" * 100)

SECTION6_REPORT_DIR = (
    NOTEBOOK50_REPORT_DIR
    / "section6"
)

SECTION6_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_POLICY_MODEL_FILE = (
    NOTEBOOK50_MODEL_DIR
    / "final_side_balanced_policy.joblib"
)

FINAL_PREPROCESSOR_FILE = (
    NOTEBOOK50_MODEL_DIR
    / "leakage_safe_preprocessor.joblib"
)

FINAL_LABEL_ENCODER_FILE = (
    NOTEBOOK50_MODEL_DIR
    / "policy_label_encoder.joblib"
)

FINAL_FEATURE_NAMES_FILE = (
    NOTEBOOK50_MODEL_DIR
    / "feature_names.json"
)

FINAL_MODEL_METADATA_FILE = (
    NOTEBOOK50_MODEL_DIR
    / "model_metadata.json"
)

MODEL_COMPARISON_FILE = (
    SECTION6_REPORT_DIR
    / "section6a_model_comparison.csv"
)

SIDE_WEIGHT_SUMMARY_FILE = (
    SECTION6_REPORT_DIR
    / "section6a_side_weight_summary.csv"
)

WEIGHT_PROFILE_FILE = (
    SECTION6_REPORT_DIR
    / "section6a_weight_profile.csv"
)

POLICY_CLASS_LOOKUP_FILE = (
    SECTION6_REPORT_DIR
    / "section6a_policy_class_lookup.csv"
)


# --------------------------------------------------------------------------------------
# Save trained objects
# --------------------------------------------------------------------------------------

joblib.dump(
    final_side_balanced_policy,
    FINAL_POLICY_MODEL_FILE,
)

joblib.dump(
    leakage_safe_preprocessor,
    FINAL_PREPROCESSOR_FILE,
)

joblib.dump(
    policy_label_encoder,
    FINAL_LABEL_ENCODER_FILE,
)


# --------------------------------------------------------------------------------------
# Save feature names
# --------------------------------------------------------------------------------------

final_feature_names = (
    leakage_safe_preprocessor
    .get_feature_names_out()
    .tolist()
)

with open(
    FINAL_FEATURE_NAMES_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        final_feature_names,
        file,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------
# Save tabular reports
# --------------------------------------------------------------------------------------

model_comparison_df.to_csv(
    MODEL_COMPARISON_FILE,
    index=False,
)

weighted_side_totals_df.to_csv(
    SIDE_WEIGHT_SUMMARY_FILE,
    index=False,
)

weight_profile_df.to_csv(
    WEIGHT_PROFILE_FILE,
    index=False,
)

policy_lookup.to_csv(
    POLICY_CLASS_LOOKUP_FILE,
    index=False,
)


# --------------------------------------------------------------------------------------
# Build metadata
# --------------------------------------------------------------------------------------

final_validation_accuracy = accuracy_score(
    y_valid,
    final_valid_predictions,
)

final_test_accuracy = accuracy_score(
    y_test,
    final_test_predictions,
)

final_validation_balanced_accuracy = (
    balanced_accuracy_score(
        y_valid,
        final_valid_predictions,
    )
)

final_test_balanced_accuracy = (
    balanced_accuracy_score(
        y_test,
        final_test_predictions,
    )
)

final_validation_macro_f1 = f1_score(
    y_valid,
    final_valid_predictions,
    average="macro",
    zero_division=0,
)

final_test_macro_f1 = f1_score(
    y_test,
    final_test_predictions,
    average="macro",
    zero_division=0,
)

model_metadata = {
    "notebook": "50_side_balanced_policy_fine_tuning",
    "stage": "SIDE_BALANCED_POLICY_TRAINING",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "random_seed": RANDOM_SEED,
    "model_type": type(
        final_side_balanced_policy
    ).__name__,
    "model_parameters": (
        final_side_balanced_policy
        .get_params()
    ),
    "training_examples": int(
        len(X_train_safe)
    ),
    "validation_examples": int(
        len(X_valid_safe)
    ),
    "test_examples": int(
        len(X_test_safe)
    ),
    "training_scenarios": int(
        groups_train.nunique()
    ),
    "validation_scenarios": int(
        groups_valid.nunique()
    ),
    "test_scenarios": int(
        groups_test.nunique()
    ),
    "policy_class_count": int(
        len(policy_classes)
    ),
    "policy_classes": policy_classes,
    "input_feature_columns": (
        LEAKAGE_SAFE_FEATURE_COLUMNS
    ),
    "encoded_feature_count": int(
        len(final_feature_names)
    ),
    "variant_name_removed": True,
    "side_balancing_enabled": True,
    "combined_weight_mean": float(
        combined_weights_train.mean()
    ),
    "relative_weighted_side_gap": float(
        relative_weight_gap
    ),
    "validation_accuracy": float(
        final_validation_accuracy
    ),
    "validation_balanced_accuracy": float(
        final_validation_balanced_accuracy
    ),
    "validation_macro_f1": float(
        final_validation_macro_f1
    ),
    "test_accuracy": float(
        final_test_accuracy
    ),
    "test_balanced_accuracy": float(
        final_test_balanced_accuracy
    ),
    "test_macro_f1": float(
        final_test_macro_f1
    ),
    "source_dataset": str(
        POLICY_DATASET_FILE
    ),
    "model_file": str(
        FINAL_POLICY_MODEL_FILE
    ),
    "preprocessor_file": str(
        FINAL_PREPROCESSOR_FILE
    ),
    "label_encoder_file": str(
        FINAL_LABEL_ENCODER_FILE
    ),
    "feature_names_file": str(
        FINAL_FEATURE_NAMES_FILE
    ),
    "next_stage": "POLICY_ARTIFACT_VALIDATION",
}

with open(
    FINAL_MODEL_METADATA_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_metadata,
        file,
        indent=2,
        ensure_ascii=False,
        default=str,
    )


# --------------------------------------------------------------------------------------
# Artifact inventory
# --------------------------------------------------------------------------------------

section6a_artifacts = {
    "final_policy_model":
        FINAL_POLICY_MODEL_FILE,

    "preprocessor":
        FINAL_PREPROCESSOR_FILE,

    "label_encoder":
        FINAL_LABEL_ENCODER_FILE,

    "feature_names":
        FINAL_FEATURE_NAMES_FILE,

    "model_metadata":
        FINAL_MODEL_METADATA_FILE,

    "model_comparison":
        MODEL_COMPARISON_FILE,

    "side_weight_summary":
        SIDE_WEIGHT_SUMMARY_FILE,

    "weight_profile":
        WEIGHT_PROFILE_FILE,

    "policy_class_lookup":
        POLICY_CLASS_LOOKUP_FILE,
}

artifact_rows = []

for artifact_name, artifact_path in (
    section6a_artifacts.items()
):
    artifact_rows.append(
        {
            "artifact_name": artifact_name,
            "path": str(artifact_path),
            "exists": artifact_path.exists(),
            "is_file": artifact_path.is_file(),
            "size_bytes": (
                artifact_path.stat().st_size
                if artifact_path.exists()
                else 0
            ),
        }
    )

section6a_artifact_inventory_df = (
    pd.DataFrame(
        artifact_rows
    )
)

section6a_artifact_inventory_df[
    "nonempty"
] = (
    section6a_artifact_inventory_df[
        "size_bytes"
    ] > 0
)

print()
print("EXPORTED ARTIFACTS")
print("-" * 100)

display(
    section6a_artifact_inventory_df
)

assert (
    section6a_artifact_inventory_df[
        "exists"
    ].all()
)

assert (
    section6a_artifact_inventory_df[
        "is_file"
    ].all()
)

assert (
    section6a_artifact_inventory_df[
        "nonempty"
    ].all()
)

print()
print("SAVED MODEL ARTIFACTS")
print("-" * 100)

for artifact_path in (
    section6a_artifacts.values()
):
    print(artifact_path)

print()
print("✅ SECTION 6A FINAL POLICY ARTIFACT EXPORT PASSED")

SECTION 6A — EXPORT FINAL POLICY ARTIFACTS

EXPORTED ARTIFACTS
----------------------------------------------------------------------------------------------------


,artifact_name,path,exists,is_file,size_bytes,nonempty
0,final_policy_model,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\final_side_balanced_policy.joblib,True,True,2754409,True
1,preprocessor,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\leakage_safe_preprocessor.joblib,True,True,4861,True
2,label_encoder,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\policy_label_encoder.joblib,True,True,543,True
3,feature_names,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\feature_names.json,True,True,637,True
4,model_metadata,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\model_metadata.json,True,True,2555,True
5,model_comparison,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_model_comparison.csv,True,True,303,True
6,side_weight_summary,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_side_weight_summar...,True,True,170,True
7,weight_profile,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_weight_profile.csv,True,True,320,True
8,policy_class_lookup,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_policy_class_looku...,True,True,98,True



SAVED MODEL ARTIFACTS
----------------------------------------------------------------------------------------------------
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\final_side_balanced_policy.joblib
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\leakage_safe_preprocessor.joblib
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\policy_label_encoder.joblib
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\feature_names.json
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\model_metadata.json
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_model_comparison.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_side_weight_summary.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_weight_profil

## Section 6B — Reload and Prediction Smoke Test

In [21]:
# ======================================================================================
# SECTION 6B — RELOAD AND DEPLOYMENT SMOKE TEST
# ======================================================================================

print("=" * 100)
print("SECTION 6B — RELOAD AND DEPLOYMENT SMOKE TEST")
print("=" * 100)

reloaded_policy_model = joblib.load(
    FINAL_POLICY_MODEL_FILE
)

reloaded_preprocessor = joblib.load(
    FINAL_PREPROCESSOR_FILE
)

reloaded_label_encoder = joblib.load(
    FINAL_LABEL_ENCODER_FILE
)

with open(
    FINAL_FEATURE_NAMES_FILE,
    "r",
    encoding="utf-8",
) as file:
    reloaded_feature_names = json.load(
        file
    )

with open(
    FINAL_MODEL_METADATA_FILE,
    "r",
    encoding="utf-8",
) as file:
    reloaded_model_metadata = json.load(
        file
    )


# --------------------------------------------------------------------------------------
# Select smoke-test rows
# --------------------------------------------------------------------------------------

SMOKE_TEST_SIZE = min(
    20,
    len(X_test_safe),
)

deployment_smoke_input = (
    X_test_safe
    .head(
        SMOKE_TEST_SIZE
    )
    .copy()
)

deployment_smoke_expected = (
    final_test_predictions[
        :SMOKE_TEST_SIZE
    ]
)

deployment_smoke_processed = (
    reloaded_preprocessor.transform(
        deployment_smoke_input
    )
)

deployment_smoke_predictions = (
    reloaded_policy_model.predict(
        deployment_smoke_processed
    )
)

deployment_smoke_probabilities = (
    reloaded_policy_model.predict_proba(
        deployment_smoke_processed
    )
)

deployment_smoke_move_names = (
    reloaded_label_encoder
    .inverse_transform(
        deployment_smoke_predictions
    )
)

expected_smoke_move_names = (
    policy_label_encoder
    .inverse_transform(
        deployment_smoke_expected
    )
)


# --------------------------------------------------------------------------------------
# Build smoke-test report
# --------------------------------------------------------------------------------------

smoke_test_df = pd.DataFrame(
    {
        "row_number": np.arange(
            SMOKE_TEST_SIZE
        ),
        "expected_label_id":
            deployment_smoke_expected,
        "reloaded_label_id":
            deployment_smoke_predictions,
        "expected_move":
            expected_smoke_move_names,
        "reloaded_move":
            deployment_smoke_move_names,
        "prediction_match": (
            deployment_smoke_expected
            == deployment_smoke_predictions
        ),
        "maximum_probability": (
            deployment_smoke_probabilities
            .max(axis=1)
        ),
        "probability_sum": (
            deployment_smoke_probabilities
            .sum(axis=1)
        ),
    }
)

print()
print("RELOADED ARTIFACT PROFILE")
print("-" * 100)

print(
    f"Model type             : "
    f"{type(reloaded_policy_model).__name__}"
)

print(
    f"Encoded feature count : "
    f"{deployment_smoke_processed.shape[1]}"
)

print(
    f"Policy classes         : "
    f"{list(reloaded_label_encoder.classes_)}"
)

print()
print("DEPLOYMENT SMOKE TEST")
print("-" * 100)

display(
    smoke_test_df
)

assert (
    deployment_smoke_processed.shape[1]
    == len(reloaded_feature_names)
)

assert list(
    reloaded_label_encoder.classes_
) == policy_classes

assert np.array_equal(
    deployment_smoke_expected,
    deployment_smoke_predictions,
)

assert (
    smoke_test_df[
        "prediction_match"
    ].all()
)

assert np.allclose(
    smoke_test_df[
        "probability_sum"
    ],
    1.0,
)

assert (
    deployment_smoke_probabilities
    >= 0
).all()

assert (
    deployment_smoke_probabilities
    <= 1
).all()

assert (
    reloaded_model_metadata[
        "next_stage"
    ]
    == "POLICY_ARTIFACT_VALIDATION"
)

SECTION6B_SMOKE_TEST_FILE = (
    SECTION6_REPORT_DIR
    / "section6b_deployment_smoke_test.csv"
)

smoke_test_df.to_csv(
    SECTION6B_SMOKE_TEST_FILE,
    index=False,
)

print()
print("SAVED SMOKE TEST")
print("-" * 100)

print(
    SECTION6B_SMOKE_TEST_FILE
)

print()
print("✅ SECTION 6B RELOAD AND DEPLOYMENT SMOKE TEST PASSED")

SECTION 6B — RELOAD AND DEPLOYMENT SMOKE TEST

RELOADED ARTIFACT PROFILE
----------------------------------------------------------------------------------------------------
Model type             : RandomForestClassifier
Encoded feature count : 18
Policy classes         : ['Ascension', 'Bind Down', 'Live Coal', 'Pass', 'Quick Attack', 'Tuck Tail']

DEPLOYMENT SMOKE TEST
----------------------------------------------------------------------------------------------------


,row_number,expected_label_id,reloaded_label_id,expected_move,reloaded_move,prediction_match,maximum_probability,probability_sum
0,0,0,0,Ascension,Ascension,True,1.0,1.0
1,1,1,1,Bind Down,Bind Down,True,1.0,1.0
2,2,1,1,Bind Down,Bind Down,True,1.0,1.0
3,3,4,4,Quick Attack,Quick Attack,True,1.0,1.0
4,4,0,0,Ascension,Ascension,True,1.0,1.0
5,5,4,4,Quick Attack,Quick Attack,True,1.0,1.0
6,6,0,0,Ascension,Ascension,True,1.0,1.0
7,7,3,3,Pass,Pass,True,1.0,1.0
8,8,0,0,Ascension,Ascension,True,1.0,1.0
9,9,4,4,Quick Attack,Quick Attack,True,1.0,1.0



SAVED SMOKE TEST
----------------------------------------------------------------------------------------------------
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6b_deployment_smoke_test.csv

✅ SECTION 6B RELOAD AND DEPLOYMENT SMOKE TEST PASSED


# Section 7 — Final Validation and Notebook 50 Handoff

## This section performs final validation across all Notebook 50 outputs.

#### It verifies:

- required model artifacts exist,
- exported files are nonempty,
- the model reload test passed,
- feature counts and class mappings are consistent,
- side balancing succeeded,
- validation and test metrics are available,
- the notebook is ready for downstream simulator integration.

## Section 7 — Final Validation and Notebook 50 Handoff

In [22]:
# ======================================================================================
# SECTION 7A — FINAL VALIDATION AND NOTEBOOK 50 HANDOFF
# ======================================================================================

print("=" * 100)
print("SECTION 7A — FINAL VALIDATION AND NOTEBOOK 50 HANDOFF")
print("=" * 100)

SECTION7_REPORT_DIR = (
    NOTEBOOK50_REPORT_DIR
    / "section7"
)

SECTION7_REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SECTION7_ARTIFACT_INVENTORY_FILE = (
    SECTION7_REPORT_DIR
    / "section7a_artifact_inventory.csv"
)

SECTION7_VALIDATION_CHECKS_FILE = (
    SECTION7_REPORT_DIR
    / "section7a_validation_checks.csv"
)

SECTION7_FINAL_PROFILE_FILE = (
    SECTION7_REPORT_DIR
    / "section7a_final_model_profile.csv"
)

SECTION7_HANDOFF_MANIFEST_FILE = (
    SECTION7_REPORT_DIR
    / "section7a_handoff_manifest.json"
)

SECTION7_FINAL_SUMMARY_FILE = (
    SECTION7_REPORT_DIR
    / "section7a_final_summary.json"
)


# --------------------------------------------------------------------------------------
# Artifact inventory
# --------------------------------------------------------------------------------------

required_notebook50_artifacts = {
    "final_policy_model":
        FINAL_POLICY_MODEL_FILE,

    "preprocessor":
        FINAL_PREPROCESSOR_FILE,

    "label_encoder":
        FINAL_LABEL_ENCODER_FILE,

    "feature_names":
        FINAL_FEATURE_NAMES_FILE,

    "model_metadata":
        FINAL_MODEL_METADATA_FILE,

    "model_comparison":
        MODEL_COMPARISON_FILE,

    "side_weight_summary":
        SIDE_WEIGHT_SUMMARY_FILE,

    "weight_profile":
        WEIGHT_PROFILE_FILE,

    "policy_class_lookup":
        POLICY_CLASS_LOOKUP_FILE,

    "deployment_smoke_test":
        SECTION6B_SMOKE_TEST_FILE,
}

artifact_inventory_rows = []

for artifact_name, artifact_path in (
    required_notebook50_artifacts.items()
):
    artifact_inventory_rows.append(
        {
            "artifact_name": artifact_name,
            "path": str(artifact_path),
            "exists": artifact_path.exists(),
            "is_file": artifact_path.is_file(),
            "size_bytes": (
                artifact_path.stat().st_size
                if artifact_path.exists()
                else 0
            ),
        }
    )

section7_artifact_inventory_df = pd.DataFrame(
    artifact_inventory_rows
)

section7_artifact_inventory_df["nonempty"] = (
    section7_artifact_inventory_df[
        "size_bytes"
    ] > 0
)

print()
print("ARTIFACT INVENTORY")
print("-" * 100)

display(
    section7_artifact_inventory_df
)


# --------------------------------------------------------------------------------------
# Validation checks
# --------------------------------------------------------------------------------------

validation_checks = []

def add_validation_check(
    check_name,
    passed,
    value="",
    expected="",
    details="",
):
    validation_checks.append(
        {
            "check": check_name,
            "passed": bool(passed),
            "value": value,
            "expected": expected,
            "details": details,
        }
    )


add_validation_check(
    "all_required_artifacts_exist",
    section7_artifact_inventory_df[
        "exists"
    ].all(),
    int(
        (~section7_artifact_inventory_df[
            "exists"
        ]).sum()
    ),
    0,
)

add_validation_check(
    "all_required_artifacts_nonempty",
    section7_artifact_inventory_df[
        "nonempty"
    ].all(),
    int(
        (~section7_artifact_inventory_df[
            "nonempty"
        ]).sum()
    ),
    0,
)

add_validation_check(
    "model_reload_prediction_match",
    smoke_test_df[
        "prediction_match"
    ].all(),
    int(
        smoke_test_df[
            "prediction_match"
        ].sum()
    ),
    len(smoke_test_df),
)

add_validation_check(
    "probability_rows_sum_to_one",
    np.allclose(
        smoke_test_df[
            "probability_sum"
        ],
        1.0,
    ),
    float(
        smoke_test_df[
            "probability_sum"
        ].mean()
    ),
    1.0,
)

add_validation_check(
    "encoded_feature_count_matches",
    (
        deployment_smoke_processed.shape[1]
        == len(reloaded_feature_names)
    ),
    deployment_smoke_processed.shape[1],
    len(reloaded_feature_names),
)

add_validation_check(
    "policy_classes_match",
    list(
        reloaded_label_encoder.classes_
    ) == policy_classes,
    len(
        reloaded_label_encoder.classes_
    ),
    len(policy_classes),
)

add_validation_check(
    "side_weight_gap_valid",
    relative_weight_gap < 0.001,
    float(relative_weight_gap),
    "< 0.001",
)

add_validation_check(
    "combined_weight_mean_valid",
    np.isclose(
        combined_weights_train.mean(),
        1.0,
    ),
    float(
        combined_weights_train.mean()
    ),
    1.0,
)

add_validation_check(
    "validation_predictions_complete",
    len(final_valid_predictions)
    == len(y_valid),
    len(final_valid_predictions),
    len(y_valid),
)

add_validation_check(
    "test_predictions_complete",
    len(final_test_predictions)
    == len(y_test),
    len(final_test_predictions),
    len(y_test),
)

add_validation_check(
    "validation_accuracy_finite",
    np.isfinite(
        final_validation_accuracy
    ),
    float(
        final_validation_accuracy
    ),
    "finite",
)

add_validation_check(
    "test_accuracy_finite",
    np.isfinite(
        final_test_accuracy
    ),
    float(
        final_test_accuracy
    ),
    "finite",
)

add_validation_check(
    "variant_name_removed",
    "variant_name"
    not in LEAKAGE_SAFE_FEATURE_COLUMNS,
    "variant_name"
    not in LEAKAGE_SAFE_FEATURE_COLUMNS,
    True,
)

add_validation_check(
    "both_sides_present_in_training",
    train_side_series.nunique() == 2,
    train_side_series.nunique(),
    2,
)

section7_validation_checks_df = pd.DataFrame(
    validation_checks
)

print()
print("FINAL VALIDATION CHECKS")
print("-" * 100)

display(
    section7_validation_checks_df
)

failed_checks_df = section7_validation_checks_df[
    ~section7_validation_checks_df[
        "passed"
    ]
]

assert failed_checks_df.empty, (
    "Notebook 50 final validation failed: "
    f"{failed_checks_df['check'].tolist()}"
)


# --------------------------------------------------------------------------------------
# Final model profile
# --------------------------------------------------------------------------------------

final_model_profile_df = pd.DataFrame(
    {
        "metric": [
            "training_examples",
            "validation_examples",
            "test_examples",
            "training_scenarios",
            "validation_scenarios",
            "test_scenarios",
            "policy_class_count",
            "encoded_feature_count",
            "validation_accuracy",
            "validation_balanced_accuracy",
            "validation_macro_f1",
            "test_accuracy",
            "test_balanced_accuracy",
            "test_macro_f1",
            "relative_weighted_side_gap",
            "combined_weight_mean",
            "smoke_test_rows",
            "validation_checks_passed",
            "validation_checks_failed",
        ],
        "value": [
            len(X_train_safe),
            len(X_valid_safe),
            len(X_test_safe),
            groups_train.nunique(),
            groups_valid.nunique(),
            groups_test.nunique(),
            len(policy_classes),
            len(final_feature_names),
            final_validation_accuracy,
            final_validation_balanced_accuracy,
            final_validation_macro_f1,
            final_test_accuracy,
            final_test_balanced_accuracy,
            final_test_macro_f1,
            relative_weight_gap,
            combined_weights_train.mean(),
            len(smoke_test_df),
            int(
                section7_validation_checks_df[
                    "passed"
                ].sum()
            ),
            int(
                (~section7_validation_checks_df[
                    "passed"
                ]).sum()
            ),
        ],
    }
)

print()
print("FINAL MODEL PROFILE")
print("-" * 100)

display(
    final_model_profile_df
)


# --------------------------------------------------------------------------------------
# Handoff manifest
# --------------------------------------------------------------------------------------

notebook50_handoff_manifest = {
    "notebook":
        "50_side_balanced_policy_fine_tuning",

    "status":
        "READY_FOR_SIMULATOR_INTEGRATION",

    "completed_sections": [
        "1A",
        "1B",
        "1C",
        "2A",
        "2B",
        "2C",
        "3A",
        "3B",
        "4A",
        "4B",
        "4C",
        "4D",
        "4E",
        "5A",
        "5B",
        "6A",
        "6B",
        "7A",
    ],

    "training_examples":
        int(len(X_train_safe)),

    "validation_examples":
        int(len(X_valid_safe)),

    "test_examples":
        int(len(X_test_safe)),

    "policy_class_count":
        int(len(policy_classes)),

    "policy_classes":
        policy_classes,

    "encoded_feature_count":
        int(len(final_feature_names)),

    "variant_name_removed":
        True,

    "side_balancing_enabled":
        True,

    "relative_weighted_side_gap":
        float(relative_weight_gap),

    "validation_accuracy":
        float(final_validation_accuracy),

    "validation_balanced_accuracy":
        float(
            final_validation_balanced_accuracy
        ),

    "validation_macro_f1":
        float(final_validation_macro_f1),

    "test_accuracy":
        float(final_test_accuracy),

    "test_balanced_accuracy":
        float(
            final_test_balanced_accuracy
        ),

    "test_macro_f1":
        float(final_test_macro_f1),

    "validation_checks_passed":
        int(
            section7_validation_checks_df[
                "passed"
            ].sum()
        ),

    "validation_checks_failed":
        int(
            (~section7_validation_checks_df[
                "passed"
            ]).sum()
        ),

    "model_file":
        str(FINAL_POLICY_MODEL_FILE),

    "preprocessor_file":
        str(FINAL_PREPROCESSOR_FILE),

    "label_encoder_file":
        str(FINAL_LABEL_ENCODER_FILE),

    "feature_names_file":
        str(FINAL_FEATURE_NAMES_FILE),

    "model_metadata_file":
        str(FINAL_MODEL_METADATA_FILE),

    "next_notebook":
        "Notebook 51 — Policy Simulator Integration",

    "next_stage":
        "POLICY_SIMULATOR_INTEGRATION",
}

notebook50_final_summary = {
    "final_status":
        notebook50_handoff_manifest[
            "status"
        ],

    "training_examples":
        notebook50_handoff_manifest[
            "training_examples"
        ],

    "validation_examples":
        notebook50_handoff_manifest[
            "validation_examples"
        ],

    "test_examples":
        notebook50_handoff_manifest[
            "test_examples"
        ],

    "policy_class_count":
        notebook50_handoff_manifest[
            "policy_class_count"
        ],

    "encoded_feature_count":
        notebook50_handoff_manifest[
            "encoded_feature_count"
        ],

    "validation_checks_passed":
        notebook50_handoff_manifest[
            "validation_checks_passed"
        ],

    "validation_checks_failed":
        notebook50_handoff_manifest[
            "validation_checks_failed"
        ],

    "next_notebook":
        notebook50_handoff_manifest[
            "next_notebook"
        ],

    "next_stage":
        notebook50_handoff_manifest[
            "next_stage"
        ],
}


# --------------------------------------------------------------------------------------
# Save final reports
# --------------------------------------------------------------------------------------

section7_artifact_inventory_df.to_csv(
    SECTION7_ARTIFACT_INVENTORY_FILE,
    index=False,
)

section7_validation_checks_df.to_csv(
    SECTION7_VALIDATION_CHECKS_FILE,
    index=False,
)

final_model_profile_df.to_csv(
    SECTION7_FINAL_PROFILE_FILE,
    index=False,
)

with open(
    SECTION7_HANDOFF_MANIFEST_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        notebook50_handoff_manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )

with open(
    SECTION7_FINAL_SUMMARY_FILE,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        notebook50_final_summary,
        file,
        indent=2,
        ensure_ascii=False,
    )


print()
print("NOTEBOOK 50 HANDOFF SUMMARY")
print("-" * 100)

print(
    f"Final status             : "
    f"{notebook50_handoff_manifest['status']}"
)

print(
    f"Training examples        : "
    f"{notebook50_handoff_manifest['training_examples']}"
)

print(
    f"Validation examples      : "
    f"{notebook50_handoff_manifest['validation_examples']}"
)

print(
    f"Test examples            : "
    f"{notebook50_handoff_manifest['test_examples']}"
)

print(
    f"Policy classes           : "
    f"{notebook50_handoff_manifest['policy_class_count']}"
)

print(
    f"Encoded features         : "
    f"{notebook50_handoff_manifest['encoded_feature_count']}"
)

print(
    f"Validation checks passed : "
    f"{notebook50_handoff_manifest['validation_checks_passed']}"
)

print(
    f"Validation checks failed : "
    f"{notebook50_handoff_manifest['validation_checks_failed']}"
)

print(
    f"Next notebook            : "
    f"{notebook50_handoff_manifest['next_notebook']}"
)

print(
    f"Next stage               : "
    f"{notebook50_handoff_manifest['next_stage']}"
)

print()
print("SAVED SECTION 7 REPORTS")
print("-" * 100)

print(SECTION7_ARTIFACT_INVENTORY_FILE)
print(SECTION7_VALIDATION_CHECKS_FILE)
print(SECTION7_FINAL_PROFILE_FILE)
print(SECTION7_HANDOFF_MANIFEST_FILE)
print(SECTION7_FINAL_SUMMARY_FILE)

print()
print("✅ SECTION 7A FINAL VALIDATION PASSED")
print()
print("🎉 NOTEBOOK 50 COMPLETE — READY FOR NOTEBOOK 51")

SECTION 7A — FINAL VALIDATION AND NOTEBOOK 50 HANDOFF

ARTIFACT INVENTORY
----------------------------------------------------------------------------------------------------


,artifact_name,path,exists,is_file,size_bytes,nonempty
0,final_policy_model,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\final_side_balanced_policy.joblib,True,True,2754409,True
1,preprocessor,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\leakage_safe_preprocessor.joblib,True,True,4861,True
2,label_encoder,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\policy_label_encoder.joblib,True,True,543,True
3,feature_names,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\feature_names.json,True,True,637,True
4,model_metadata,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\models\notebook50\model_metadata.json,True,True,2555,True
5,model_comparison,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_model_comparison.csv,True,True,303,True
6,side_weight_summary,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_side_weight_summar...,True,True,170,True
7,weight_profile,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_weight_profile.csv,True,True,320,True
8,policy_class_lookup,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6a_policy_class_looku...,True,True,98,True
9,deployment_smoke_test,D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section6\section6b_deployment_smoke_t...,True,True,945,True



FINAL VALIDATION CHECKS
----------------------------------------------------------------------------------------------------


,check,passed,value,expected,details
0,all_required_artifacts_exist,True,0,0,
1,all_required_artifacts_nonempty,True,0,0,
2,model_reload_prediction_match,True,20,20,
3,probability_rows_sum_to_one,True,1.0,1.0,
4,encoded_feature_count_matches,True,18,18,
5,policy_classes_match,True,6,6,
6,side_weight_gap_valid,True,0.0,< 0.001,
7,combined_weight_mean_valid,True,1.0,1.0,
8,validation_predictions_complete,True,110,110,
9,test_predictions_complete,True,110,110,



FINAL MODEL PROFILE
----------------------------------------------------------------------------------------------------


,metric,value
0,training_examples,8.400000e+02
1,validation_examples,1.100000e+02
2,test_examples,1.100000e+02
3,training_scenarios,8.400000e+01
4,validation_scenarios,1.100000e+01
5,test_scenarios,1.100000e+01
6,policy_class_count,6.000000e+00
7,encoded_feature_count,1.800000e+01
8,validation_accuracy,1.000000e+00
9,validation_balanced_accuracy,1.000000e+00



NOTEBOOK 50 HANDOFF SUMMARY
----------------------------------------------------------------------------------------------------
Final status             : READY_FOR_SIMULATOR_INTEGRATION
Training examples        : 840
Validation examples      : 110
Test examples            : 110
Policy classes           : 6
Encoded features         : 18
Validation checks passed : 14
Validation checks failed : 0
Next notebook            : Notebook 51 — Policy Simulator Integration
Next stage               : POLICY_SIMULATOR_INTEGRATION

SAVED SECTION 7 REPORTS
----------------------------------------------------------------------------------------------------
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section7\section7a_artifact_inventory.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section7\section7a_validation_checks.csv
D:\02_AI_and_Data\Kaggle-AI-Agents\PTCG_AI_Battle_Challenge\reports\notebook50\section7\section7a_final_mod